In [1]:
# import libraries
import numpy as np
import sys
sys.path.append("/Users/jfoley19/Code/SCQED-PCQED/src/")
import psi4
from helper_PFCI import PFHamiltonianGenerator
np.set_printoptions(threshold=sys.maxsize)
psi4.core.set_output_file('output.dat', False)
import time
import json
import matplotlib.pyplot as plt

In [2]:
# read data from .npy files for formaldehyde casci(8,8) calculations

# !!! Change this to the correct path on your computer!
#npy_folder = "/Users/rmandern/code/SCQED-PCQED/OH/"
npy_folder = "/Users/jfoley19/Code/data_repository/NPY_Files/OHminus/"

# these file names should still be good
E0_npy_file = npy_folder + "OH_minus_r_0.9_displaced_0Ang_631g_fci_50_states_Energies.npy"
Mu0_npy_file = npy_folder + "OH_minus_r_0.9_displaced_0Ang_631g_fci_50_states_Dipoles.npy"
E20_npy_file = npy_folder + "OH_minus_r_0.9_displaced_20Ang_631g_fci_50_states_Energies.npy"
Mu20_npy_file = npy_folder + "OH_minus_r_0.9_displaced_20Ang_631g_fci_50_states_Dipoles.npy"

# store energy eigenvalues in E_array
E0_array = np.load(E0_npy_file)
E20_array = np.load(E20_npy_file)
# store dipole matrix elements in Mu_array
Mu0_array = np.load(Mu0_npy_file)
Mu20_array = np.load(Mu20_npy_file)


print(np.shape(E0_array))
print(np.shape(Mu0_array))
# print(E_array)

(50,)
(50, 50, 3)


In [3]:
# setup basic arguments to create an instance of the PFHamiltonianGenerator class
mol_str = """
   O   0.000000000000  0.0 0.0
    H   0.000000000000  0.0 0.9       
    -1  1
    symmetry c1
    no_reorient
    no_com
"""


options_dict = {
    "basis": "sto-3g",
    "scf_type": "pk",
    "e_convergence": 1e-10,
    "d_convergence": 1e-10,
}


cavity_free_dict = {
    'omega_value' : 0,
    'lambda_vector' : np.array([0, 0, 0.0]),
    'ci_level' : 'fci',   
    'full_diagonalization' : True,
    'number_of_photons' : 0, 
}

# create the instance of our PFHamiltonianGenerator class
origin_instance = PFHamiltonianGenerator(mol_str, options_dict, cavity_free_dict)
displaced_instance = PFHamiltonianGenerator(mol_str, options_dict, cavity_free_dict)


Start SCF iterations:

Canonical RHF One-electron energy = -118.7531098436234771
CQED-RHF One-electron energy      = -118.7531098436234771
Nuclear repulsion energy          = 4.7037974281777775
Dipole energy                     = 0.0000000000000000
SCF Iteration   1: Energy = -74.0389663712315524   dE = -7.40390E+01   dRMS = 2.51528E-15
SCF Iteration   2: Energy = -74.0389663712315524   dE =  0.00000E+00   dRMS = 4.28853E-15
Total time for SCF iterations: 0.000 seconds 

QED-RHF   energy: -74.03896637 hartree
Psi4  SCF energy: -74.03896637 hartree
 Completed QED-RHF in 0.21003508567810059 seconds
 Completed 1HSO Build in 5.1021575927734375e-05 seconds
 Completed ERI Build in 0.0011448860168457031 seconds 
 Completed 2D build in 0.0001480579376220703 seconds
 Completed 1G build in 1.1920928955078125e-05 seconds
 Completed the Dipole Matrix Build in 3.409385681152344e-05 seconds
 Completed determinant list in 0.00011396408081054688 seconds 
 Completed constant offset matrix in 1.8835067

In [4]:

N_el = 50
N_ph = [2, 3, 5, 9, 17, 33]
omega = 0.21927747
lambda_vector = np.array([0, 0, 0.05])


N_l = len(N_ph)

origin_energies = []
displaced_energies = []
error = []

for i in N_ph:
    _pcqed = np.zeros(N_el * i)
    origin_instance.fast_build_pcqed_pf_hamiltonian(N_el, i, omega, lambda_vector, E0_array, Mu0_array)
    displaced_instance.fast_build_pcqed_pf_hamiltonian(N_el, i, omega, lambda_vector, E20_array, Mu20_array)
    or_en = origin_instance.PCQED_pf_eigs[0]
    di_en = displaced_instance.PCQED_pf_eigs[0]
    or_er = di_en - or_en
    origin_energies.append(or_en)
    displaced_energies.append(di_en)
    error.append(or_er)
    



In [6]:
print(N_ph)
print(error)

[2, 3, 5, 9, 17, 33]
[1.1973436651193907, 0.8846499098818441, 0.5030561682658288, 0.1517958442292695, 0.003426788843057693, 1.298019469686551e-10]


In [11]:



# set up base dictionary - some of this will be updated with each calculation
dictionary = {   
    "molecule": {
        "molecule_name": "OH-",
        "geometry": [
            "\nO       \nH            1    0.9\n1 1\nsymmetry c1\n"
        ],
        "symbols": [
            "O",
            "H"
        ],
        "displacement" : 20,
    },
    "model" : {
        "orbital_basis" : "6-31G",
        "photon_basis" : "photon_number_basis", 
        "method" : "PCQED",
        "number_of_photon_states" : N_ph,
        "omega" : '0.21927747',
        "lambda" : [
            0.0,
            0.0,
            0.05
        ],
    },
    "return_result_origin" : origin_energies,
    "return_result_displaced" : displaced_energies,
    "origin_dependent_error" : error
}


# function to generate file names based on system details
def generate_file_name(dic):
    
    file_name = dic["molecule"]["molecule_name"] + "_"
    file_name += str(dictionary["model"]["orbital_basis"]) + "_"
    file_name += str(dictionary["model"]["method"]) + "_"
    file_name += str(dictionary["model"]["lambda"][2]) + "_"
    file_name += str(dictionary["model"]["omega"]) + ".json"
    return file_name



# write to json file
file_name = generate_file_name(dictionary)
json_object = json.dumps(dictionary, indent=4)
with open(file_name, "w") as outfile:
    outfile.write(json_object)

In [ ]:
# N_p = (2,4,6,8,10,12,14,16,32)
# error = (1.1973436651194191,0.6655110228316659,0.379202982740523,0.20924627270564145,0.10774471275873054,0.04985546665081131,0.019858190500841033,0.0065112765347805635,5.063185426479322e-10)


import os
import glob
import json
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib import rcParams
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 12


COLOUR1 = "firebrick"
COLOUR2 = "darkgreen"
COLOUR3 = "royalblue"
COLOUR4 = "rebeccapurple"
COLOUR5 = 'darkorchid'
COLOUR6 = 'olivedrab'


plt.plot(N_p, error, color=COLOUR1, label="pPf" )
# plt.plot(x_pfci, fit_pfci, color=COLOUR1, linestyle="none", marker="o",  ms="8", mfc="none", label="Quartic Fit")
plt.xlabel("Number of Photonic Fock States")
plt.ylabel("Origin Dependent Error")
plt.legend()
plt.show()

In [ ]:
N_R = 2
# d_array = np.linspace(0, 20, N_R)
# N_l = len(d_array)
N_el = 50
N_ph = 2
omega = 0.21927747
lambda_vector = np.array([0, 0, 0.05])

# create an array of zeros to store the PCQED eigenvalues for each value of d
_pcqed_0 = np.zeros( N_el * N_ph)

instance.fast_build_pcqed_pf_hamiltonian(N_el, N_ph, omega, lambda_vector , E0_array, Mu0_array)      # E_array[:,ctr]: 20 energy values for fisrt displacement and so on... 
_pcqed_0 = instance.PCQED_pf_eigs


In [ ]:
# N_R = 2
# d_array = np.linspace(0, 20, N_R)
# N_l = len(d_array)
# N_el = 50
# N_ph = 2
# omega = 0.21927747
# lambda_vector = np.array([0, 0, 0.05])

# # create an array of zeros to store the PCQED eigenvalues for each value of d
# _pcqed_20 = np.zeros((N_l, N_el * N_ph))
# # loop over values of d, build Hamiltonian, capture eigenvalues
# ctr = 0
# for d in d_array:
#     instance.fast_build_pcqed_pf_hamiltonian(N_el, N_ph, omega, lambda_vector , E20_array, Mu20_array)      # E_array[:,ctr]: 20 energy values for fisrt displacement and so on... 
#     _pcqed_20[ctr, :] = instance.PCQED_pf_eigs
#     ctr += 1